# Data Cleaning Guide: `cleaned_palay.xlsx`

This beginner-friendly notebook walks through inspecting and cleaning **`cleaned_palay.xlsx`** step by step using **pandas**.

Even when a file already looks mostly clean, running a proper cleaning pipeline catches subtle issues like mixed column header types (`int` vs `str`) and hidden missing cells (`NaN`).

### Cleaning Checklist:
1. **Load & Inspect** the dataset.
2. **Standardize Column Headers** (convert integer years like `1987` to strings `'1987'`).
3. **Clean Text & Check Duplicates** in `Location`.
4. **Find & Handle Missing Values (`NaN`)** (`Batanes` in `1989` and `Maguindanao del Norte`).
5. **Validate Numbers & Save** the finalized Excel file.

## Step 1: Import Pandas and Load the Dataset
Let's import `pandas` and read `cleaned_palay.xlsx`.

In [1]:
import pandas as pd

# Load the Excel file
df = pd.read_excel('cleaned_palay.xlsx')

# Check the number of rows and columns
print('Shape (rows, columns):', df.shape)

# Preview the first 5 rows
print(df.head())

Shape (rows, columns): (82, 40)
  Location     1987      1988     1989     1990     1991     1992     1993     1994     1995     1996     1997     1998     1999     2000      2001      2002      2003      2004      2005      2006      2007      2008      2009      2010      2011      2012      2013      2014      2015      2016      2017       2018       2019       2020       2021       2022       2023       2024       2025
0     Abra  23085.0   31157.0  27509.0  25637.0  24109.0  20483.0  23871.0  24340.0  29309.0  34810.0  35039.0  31231.0  44167.0  43881.0   44647.0   43481.0   46756.0   48339.0   50605.0   52989.0   73001.0   75528.0   76302.0   77611.0   80428.0   82333.0   79444.0   75500.0   66716.0   65159.0   69477.0   62120.00   82429.69   83667.10   90434.39   76921.00   74414.98   68864.60   73327.60
1   Apayao      0.0       0.0      0.0      0.0      0.0      0.0      0.0  28592.0  27974.0  28374.0  29790.0  22057.0  26841.0  48937.0   59973.0   59843.0   60744.0   86027.

## Step 2: Standardize Column Names
When Excel has numeric year headers (`1987`, `1988`), pandas reads them as **integers** (`int`) instead of **strings** (`str`). Converting all headers to strings avoids `KeyError` issues when selecting columns later.

In [2]:
# Check header data types before fixing
print('Before:', [type(c).__name__ for c in df.columns[:5]])

# Convert all column names to clean strings
df.columns = df.columns.astype(str).str.strip()

print('After: ', [type(c).__name__ for c in df.columns[:5]])

Before: ['str', 'int', 'int', 'int', 'int']
After:  ['str', 'str', 'str', 'str', 'str']


## Step 3: Clean Province Names & Check Duplicates
Let's trim whitespace from `Location` and check that no provinces are duplicated.

In [3]:
# Strip leading and trailing spaces
df['Location'] = df['Location'].str.strip()

# Check for duplicate rows or duplicate province names
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate provinces:', df['Location'].duplicated().sum())

Duplicate rows: 0
Duplicate provinces: 0


## Step 4: Find Missing Values (`NaN`)
Let's check which provinces contain `NaN` values and inspect the exact years affected.

In [4]:
# Find rows with at least one missing value
missing_rows = df[df.isnull().any(axis=1)]
print('Provinces with missing values:', list(missing_rows['Location']))

# Inspect Batanes around 1989
print('\nBatanes (1987-1991):')
print(df[df['Location'] == 'Batanes'][['Location', '1987', '1988', '1989', '1990', '1991']])

# Inspect Maguindanao del Norte
print('\nMaguindanao del Norte (1987, 2023-2025):')
print(df[df['Location'] == 'Maguindanao del Norte'][['Location', '1987', '2023', '2024', '2025']])

Provinces with missing values: ['Batanes', 'Maguindanao del Norte']

Batanes (1987-1991):
   Location  1987  1988  1989  1990  1991
10  Batanes   0.0   0.0   NaN   0.0   0.0

Maguindanao del Norte (1987, 2023-2025):
                 Location  1987  2023       2024      2025
78  Maguindanao del Norte   NaN   NaN  266291.48  277021.0


## Step 5: Handle Missing Values
We found two spots with `NaN`:
1. **`Batanes` (`1989`)**: Surrounded by `0.0` in `1987`, `1988`, `1990`, and `1991`.
2. **`Maguindanao del Norte` (`1987–2023`)**: Newly created province before 2024.

Using `fillna(0)` fills these gaps cleanly so every cell in the table is a valid number.

In [5]:
# Fill missing values with 0
df = df.fillna(0)

# Confirm no missing values remain
print('Remaining missing values:', df.isnull().sum().sum())

Remaining missing values: 0


## Step 6: Validate Numeric Values & Save
Let's verify that all year columns are numeric, check that there are no negative values, and export the final cleaned Excel file.

In [6]:
year_cols = [col for col in df.columns if col != 'Location']

# Ensure every year column is numeric
for col in year_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Verify no negative values exist
print('Negative values count:', (df[year_cols] < 0).sum().sum())

# Save the final cleaned dataset
df.to_excel('palay_production_final.xlsx', index=False)
print('Saved successfully to palay_production_final.xlsx!')

Negative values count: 0
Saved successfully to palay_production_final.xlsx!
